# Notebook 03 — Baseline Groq & RAG avec FAISS

**Ordre pipeline :** `01` → `02` → **`03`** → `04` → `05` → `06` → `07` → `08`.

**Objectif** :
1. **Baseline** : Groq sans contexte
2. **RAG** : Groq + top-k FAISS (bi-encodeur)

**Outputs** : `baseline_predictions.json`, `rag_predictions.json`, index `models/faiss_index/`.


## 0. Montage Google Drive

In [ ]:
# Montage du Drive et définition du chemin de base du projet
from google.colab import drive
drive.mount('/content/drive')

BASE_PATH = '/content/drive/MyDrive/llm-integration-study/'

## 1. Installation des dépendances

In [ ]:
# Installation du SDK Groq, sentence-transformers, faiss-cpu et codecarbon (impact écologique)
!pip install -q groq sentence-transformers faiss-cpu codecarbon

## 2. Imports, configuration et clé API

In [ ]:
# Imports de toutes les bibliothèques nécessaires
import os
import json
import time
import getpass
import numpy as np
import faiss
from groq import Groq
from sentence_transformers import SentenceTransformer
from tqdm.notebook import tqdm

# Définition de tous les chemins
RAW_PATH        = os.path.join(BASE_PATH, 'data', 'raw')
PROCESSED_PATH  = os.path.join(BASE_PATH, 'data', 'processed')
RESULTS_PATH    = os.path.join(BASE_PATH, 'results')
FAISS_PATH      = os.path.join(BASE_PATH, 'models', 'faiss_index')

for path in [RESULTS_PATH, FAISS_PATH]:
    os.makedirs(path, exist_ok=True)

GROQ_MODEL  = "llama-3.1-8b-instant"
# Modèle multilingue indispensable pour un corpus majoritairement français
EMBED_MODEL = "paraphrase-multilingual-MiniLM-L12-v2"
TOP_K       = 5  # Top-5 pour couvrir les chunks pertinents épars

print("Répertoires prêts :")
print(f"  Résultats   : {RESULTS_PATH}")
print(f"  Index FAISS : {FAISS_PATH}")

In [ ]:
# Saisie sécurisée de la clé Groq (https://console.groq.com → API Keys)
api_key     = getpass.getpass("Entre ta clé Groq API : ")
groq_client = Groq(api_key=api_key)
print("Client Groq initialisé.")

## 3. Chargement des données

In [ ]:
# Chargement de train.json (corpus RAG) et test.json (questions)
def load_json(path):
    """Charge un fichier JSON avec gestion d'erreur."""
    try:
        with open(path, 'r', encoding='utf-8') as f:
            data = json.load(f)
        print(f"  Chargé : {path} ({len(data)} entrées)")
        return data
    except FileNotFoundError:
        print(f"  [ERROR] Fichier introuvable : {path}")
        print("  → Assurez-vous d'avoir exécuté 02_dataset_builder.ipynb au préalable.")
        return []
    except json.JSONDecodeError as e:
        print(f"  [ERROR] JSON invalide : {e}")
        return []

print("Chargement des datasets...")
# Documents bruts : seront découpés en chunks pour l'index FAISS
wiki_docs    = load_json(os.path.join(RAW_PATH, 'wikipedia_technique.json'))
hal_docs     = load_json(os.path.join(RAW_PATH, 'hal.json'))
lemonde_docs = load_json(os.path.join(RAW_PATH, 'lemonde.json'))
legal_docs   = load_json(os.path.join(RAW_PATH, 'code_route.json'))
raw_docs     = wiki_docs + hal_docs + lemonde_docs + legal_docs
# Questions de test
test_data    = load_json(os.path.join(PROCESSED_PATH, 'test.json'))

print(f"\nCorpus RAG (docs bruts) : {len(raw_docs)} documents")
print(f"  Wikipedia  : {len(wiki_docs)} articles")
print(f"  HAL (FR)   : {len(hal_docs)} papiers")
print(f"  Le Monde   : {len(lemonde_docs)} articles")
print(f"  Code route : {len(legal_docs)} segments (PDF, optionnel)")
print(f"Set de test : {len(test_data)} questions")

## 4. BASELINE — Groq sans contexte

In [ ]:
# Fonction d'appel Groq pour la baseline (aucun contexte injecté)
# Groq free tier : ~30 req/min → throttle 2.5s entre chaque appel
BASELINE_PROMPT = """Tu réponds en français, de façon concise et factuelle. Les questions portent sur des sujets variés (sciences, IA, actualité, droit, etc.).

Question : {question}"""

def call_groq_baseline(question, retries=4):
    """Appelle Groq sans contexte et mesure la latence.
    Retourne (predicted_answer, latency_ms, tokens_in, tokens_out).
    """
    prompt = BASELINE_PROMPT.format(question=question)
    for attempt in range(retries):
        try:
            start = time.time()
            response = groq_client.chat.completions.create(
                model=GROQ_MODEL,
                messages=[{"role": "user", "content": prompt}]
            )
            latency_ms  = round((time.time() - start) * 1000)
            answer      = response.choices[0].message.content.strip()
            tokens_in   = response.usage.prompt_tokens     if response.usage else 0
            tokens_out  = response.usage.completion_tokens if response.usage else 0
            return answer, latency_ms, tokens_in, tokens_out
        except Exception as e:
            error_str = str(e)
            if '429' in error_str or 'rate_limit' in error_str.lower():
                wait = 5 * (2 ** attempt)
                print(f"  [QUOTA] Attente {wait}s... ({attempt+1}/{retries})")
                time.sleep(wait)
            else:
                print(f"  [ERROR] {error_str[:100]}")
                time.sleep(1)
    return "", 0, 0, 0

In [ ]:
# Exécution de la baseline sur tout le test set
# Durée estimée : ~1 min (120 questions × 0.5s throttle — Groq Developer)
THROTTLE_S = 0.5  # Groq Developer — pas de limite pratique

baseline_predictions = []

for item in tqdm(test_data, desc="Baseline (sans contexte)"):
    question    = item.get('question', '')
    true_answer = item.get('answer', '')

    predicted, latency, tok_in, tok_out = call_groq_baseline(question)

    baseline_predictions.append({
        "pair_id":          item.get('pair_id', ''),
        "question":         question,
        "predicted_answer": predicted,
        "true_answer":      true_answer,
        "latency_ms":       latency,
        "tokens_in":        tok_in,
        "tokens_out":       tok_out,
        "method":           "baseline",
        "dataset_type":     item.get("dataset_type", ""),
        "question_type":    item.get("question_type", ""),
    })
    time.sleep(THROTTLE_S)

lats  = [p['latency_ms'] for p in baseline_predictions if p['latency_ms'] > 0]
t_out = [p['tokens_out'] for p in baseline_predictions]
print(f"\nBaseline terminée : {len(baseline_predictions)} prédictions")
print(f"Latence moyenne   : {np.mean(lats):.0f} ms")
print(f"Tokens générés    : moy. {np.mean(t_out):.0f} / total {sum(t_out)}")

In [ ]:
# Sauvegarde des prédictions baseline sur Drive
baseline_path = os.path.join(RESULTS_PATH, 'baseline_predictions.json')
try:
    with open(baseline_path, 'w', encoding='utf-8') as f:
        json.dump(baseline_predictions, f, ensure_ascii=False, indent=2)
    print(f"Baseline sauvegardée : {baseline_path}")
    print(f"  → {len(baseline_predictions)} prédictions, {os.path.getsize(baseline_path)/1024:.1f} Ko")
except Exception as e:
    print(f"[ERROR] Sauvegarde baseline : {e}")

## 5. RAG — Construction de l'index FAISS

In [ ]:
# Chargement du modèle d'embedding sentence-transformers
print(f"Chargement du modèle d'embedding : {EMBED_MODEL}")
embed_model = SentenceTransformer(EMBED_MODEL)
print("Modèle chargé.")

In [ ]:
# ── Chunking des documents source (config retenue par ablation) ─────────
CHUNK_SIZE = 200   # mots par chunk (meilleur compromis observé)
CHUNK_OVL  = 50    # chevauchement (meilleur compromis observé)

def chunk_document(doc, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVL):
    """Découpe le contenu d'un document en chunks chevauchants."""
    text  = doc.get('content', '').strip()
    words = text.split()
    chunks = []
    start = 0
    while start < len(words):
        end  = min(start + chunk_size, len(words))
        chunk_text = ' '.join(words[start:end])
        chunks.append({
            "text":         chunk_text,
            "doc_id":       doc.get('id', ''),
            "title":        doc.get('title', ''),
            "source":       doc.get('source', ''),
            "dataset_type": doc.get('dataset_type', ''),
            "date":         doc.get('date', ''),
            "chunk_idx":    len(chunks),
        })
        if end == len(words):
            break
        start += chunk_size - overlap
    return chunks

corpus_chunks = []
for doc in raw_docs:
    corpus_chunks.extend(chunk_document(doc))

corpus_texts = [c['text'] for c in corpus_chunks]
corpus_meta  = corpus_chunks

words_per_chunk = [len(c['text'].split()) for c in corpus_chunks]
print(f"Corpus RAG — {len(corpus_chunks)} chunks (moy. {sum(words_per_chunk)//len(words_per_chunk)} mots/chunk)")
print(f"  Wikipedia  : {sum(1 for c in corpus_chunks if c['dataset_type']=='technique')} chunks")
print(f"  HAL        : {sum(1 for c in corpus_chunks if c['dataset_type']=='multisauts')} chunks")
print(f"  Le Monde   : {sum(1 for c in corpus_chunks if c['dataset_type']=='temporel')} chunks")
print(f"  Juridique  : {sum(1 for c in corpus_chunks if c['dataset_type']=='juridique')} chunks")

# ── Embedding + index FAISS ───────────────────────────────────────────────
print("\nCalcul des embeddings...")
corpus_embeddings = embed_model.encode(
    corpus_texts,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

dim   = corpus_embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(corpus_embeddings.astype(np.float32))
print(f"Index FAISS construit : {index.ntotal} vecteurs, dimension {dim}")

In [ ]:
# Persistance de l'index FAISS et des métadonnées sur Drive
faiss_index_file = os.path.join(FAISS_PATH, 'index.faiss')
faiss_meta_file  = os.path.join(FAISS_PATH, 'metadata.json')

try:
    faiss.write_index(index, faiss_index_file)
    print(f"Index FAISS sauvegardé : {faiss_index_file}")
except Exception as e:
    print(f"[ERROR] Sauvegarde index FAISS : {e}")

try:
    with open(faiss_meta_file, 'w', encoding='utf-8') as f:
        json.dump(corpus_meta, f, ensure_ascii=False, indent=2)
    print(f"Métadonnées sauvegardées : {faiss_meta_file}")
except Exception as e:
    print(f"[ERROR] Sauvegarde métadonnées : {e}")

## 6. RAG — Inférence sur le test set

In [ ]:
# Fonctions de récupération FAISS et d'appel Groq avec contexte injecté (réponse en FR)
RAG_PROMPT_TEMPLATE = """Réponds à la question suivante EN FRANÇAIS en te basant sur les extraits fournis ci-dessous.
Les sujets peuvent être variés (sciences, intelligence artificielle, actualité, textes juridiques, etc.).
Si un extrait est en anglais ou incomplet, extrais l’information pertinente en français. Reste factuel et concis.


Contexte :
{context_block}

Question : {question}"""

def retrieve_top_k(question, k=TOP_K):
    """Encode la question et récupère les k chunks les plus proches dans FAISS."""
    q_emb = embed_model.encode(
        [question],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype(np.float32)
    scores, indices = index.search(q_emb, k)
    chunks = [corpus_meta[i] for i in indices[0] if i < len(corpus_meta)]
    return chunks, scores[0].tolist()

def call_groq_rag(question, chunks, retries=4):
    """Appelle Groq avec les chunks récupérés comme contexte, réponse en français.
    Retourne (predicted_answer, latency_ms, tokens_in, tokens_out).
    """
    context_block = "\n\n".join([
        f"[Extrait {i+1} — {c.get('title','')[:60]}] {c.get('text', '')}"
        for i, c in enumerate(chunks)
    ])
    prompt = RAG_PROMPT_TEMPLATE.format(
        context_block=context_block,
        question=question
    )
    for attempt in range(retries):
        try:
            start = time.time()
            response = groq_client.chat.completions.create(
                model=GROQ_MODEL,
                messages=[{"role": "user", "content": prompt}]
            )
            latency_ms  = round((time.time() - start) * 1000)
            answer      = response.choices[0].message.content.strip()
            tokens_in   = response.usage.prompt_tokens     if response.usage else 0
            tokens_out  = response.usage.completion_tokens if response.usage else 0
            return answer, latency_ms, tokens_in, tokens_out
        except Exception as e:
            error_str = str(e)
            if '429' in error_str or 'rate_limit' in error_str.lower():
                wait = 5 * (2 ** attempt)
                print(f"  [QUOTA] Attente {wait}s... ({attempt+1}/{retries})")
                time.sleep(wait)
            else:
                print(f"  [ERROR] {error_str[:100]}")
                time.sleep(1)
    return "", 0, 0, 0

print("Fonctions RAG prêtes. Les réponses seront générées EN FRANÇAIS.")

In [ ]:
# Exécution RAG sur tout le test set
# Durée estimée : ~1 min (120 questions × 0.5s throttle — Groq Developer)
rag_predictions = []

for item in tqdm(test_data, desc="RAG (top-5 FAISS + Groq)"):
    question    = item.get('question', '')
    true_answer = item.get('answer', '')

    chunks, scores = retrieve_top_k(question, k=TOP_K)
    predicted, latency, tok_in, tok_out = call_groq_rag(question, chunks)

    rag_predictions.append({
        "pair_id":          item.get('pair_id', ''),
        "question":         question,
        "predicted_answer": predicted,
        "true_answer":      true_answer,
        "latency_ms":       latency,
        "tokens_in":        tok_in,
        "tokens_out":       tok_out,
        "retrieved_chunks": [f"{c.get('doc_id','')}#{c.get('chunk_idx','')}" for c in chunks],
        "retrieval_scores": scores,
        "method":           "rag",
        "dataset_type":     item.get("dataset_type", ""),
        "question_type":    item.get("question_type", ""),
    })
    time.sleep(THROTTLE_S)

lats  = [p['latency_ms'] for p in rag_predictions if p['latency_ms'] > 0]
t_out = [p['tokens_out'] for p in rag_predictions]
print(f"\nRAG terminé : {len(rag_predictions)} prédictions")
print(f"Latence moyenne : {np.mean(lats):.0f} ms")
print(f"Tokens générés  : moy. {np.mean(t_out):.0f} / total {sum(t_out)}")

In [ ]:
# Sauvegarde des prédictions RAG sur Drive
rag_path = os.path.join(RESULTS_PATH, 'rag_predictions.json')
try:
    with open(rag_path, 'w', encoding='utf-8') as f:
        json.dump(rag_predictions, f, ensure_ascii=False, indent=2)
    print(f"RAG sauvegardé : {rag_path}")
    print(f"  → {len(rag_predictions)} prédictions, {os.path.getsize(rag_path)/1024:.1f} Ko")
except Exception as e:
    print(f"[ERROR] Sauvegarde RAG : {e}")

## 7. Résumé final

In [ ]:
# Affichage du résumé de ce qui a été produit
baseline_latencies = [p['latency_ms'] for p in baseline_predictions if p['latency_ms'] > 0]
rag_latencies      = [p['latency_ms'] for p in rag_predictions      if p['latency_ms'] > 0]

print("=" * 65)
print("RÉSUMÉ — Notebook 03 : Baseline & RAG")
print("=" * 65)
print(f"\n{'Méthode':<20} {'Prédictions':>12} {'Latence moy.':>14}")
print("-" * 50)
print(f"{'Baseline':<20} {len(baseline_predictions):>12}   {np.mean(baseline_latencies):.0f} ms" if baseline_latencies else f"{'Baseline':<20} {len(baseline_predictions):>12}   N/A")
print(f"{'RAG':<20} {len(rag_predictions):>12}   {np.mean(rag_latencies):.0f} ms" if rag_latencies else f"{'RAG':<20} {len(rag_predictions):>12}   N/A")

print(f"\nIndex FAISS : {index.ntotal} chunks — dim {dim} — {len(raw_docs)} docs source")
print(f"\nFichiers produits :")
for fpath in [baseline_path, rag_path, faiss_index_file, faiss_meta_file]:
    try:
        size = os.path.getsize(fpath)
        print(f"  {fpath}  ({size/1024:.1f} Ko)")
    except Exception:
        print(f"  {fpath}  (non trouvé)")

print("\nAperçu d'une prédiction RAG :")
if rag_predictions:
    s = rag_predictions[0]
    print(f"  Q        : {s['question'][:80]}")
    print(f"  Prédit   : {s['predicted_answer'][:80]}")
    print(f"  Vrai     : {s['true_answer'][:80]}")
    print(f"  Chunks   : {s['retrieved_chunks']}")
    print(f"  Scores   : {[round(x,3) for x in s['retrieval_scores']]}")

print("\n✔ Notebook 03 terminé. Lancez 04_finetuning.ipynb (nécessite un GPU).")
print("=" * 65)